In [70]:
import pandas as pd
import fitz  
import re
import io
import ipywidgets as widgets

In [71]:
# This function converts a single Likert scale value — regardless of whether
# it is stored as a number, a text label, or a combined format like "4 - Satisfied" —
# into a plain integer.
#
# Input:
#   raw_value      : a single cell value from your DataFrame (could be int, float, or string)
#   label_map      : a dictionary where keys are text labels and values are integers
#                    e.g. {"Poor": 1, "Fair": 2, "Good": 3, "Very Good": 4, "Excellent": 5}
#
# Returns:
#   An integer representing the numeric Likert score, or None if the value
#   cannot be interpreted (so you can spot and handle problem rows separately).

def convert_likert_value(raw_value, label_map):

    # If the cell is empty (pandas represents missing values as float NaN),
    # we return None immediately. We don't try to convert a missing value.
    # pd.isna() returns True for NaN, None, and similar "missing" sentinels.
    if pd.isna(raw_value):
        return None

    # Convert the value to a string so we can apply text operations consistently,
    # regardless of whether it came in as a number or text.
    # str() turns any Python value into its string representation.
    # .strip() removes leading and trailing whitespace (spaces, tabs, newlines).
    # This handles cases like "  Good  " which should match "Good".
    cleaned = str(raw_value).strip()

    # --- Attempt 1: The value is already a plain number ---
    # Some rows may already be stored as "3" or "4.0".
    # We try to convert directly to float first (not int) because "4.0" cannot
    # be passed directly to int() without going through float first.
    # int(float("4.0")) works; int("4.0") raises an error.
    try:
        return int(float(cleaned))
    except ValueError:
        # If conversion fails (e.g. the value is "Good"), Python raises a ValueError.
        # We catch it here and move on to the next attempt rather than crashing.
        pass

    # --- Attempt 2: The value is a text label like "Good" or "very good" ---
    # We normalize to lowercase so "Good", "GOOD", and "good" all match the same key.
    # The label_map keys should also be lowercase for this to work — we handle that below.
    normalized = cleaned.lower()

    # We also build a lowercase version of the label_map for comparison.
    # Dictionary comprehension: for each key-value pair in label_map,
    # create a new pair where the key is lowercased.
    # This means the caller does not have to worry about capitalization in their map.
    lowercase_map = {key.lower(): value for key, value in label_map.items()}

    if normalized in lowercase_map:
        # The text label matched a known key — return its mapped integer.
        return lowercase_map[normalized]

    # --- Attempt 3: The value is a combined format like "4 - Satisfied" or "4: Good" ---
    # We check if the first character is a digit. If it is, we try to extract
    # just the leading number before any separator character.
    # Common separators are " - ", ": ", or just a space followed by text.
    if cleaned[0].isdigit():
        # .split() with no argument splits on any whitespace.
        # We take the first piece [0], which would be "4" in "4 - Satisfied".
        # We strip any trailing punctuation like "." or ":" from that piece.
        leading_part = cleaned.split()[0].rstrip(".-:")
        try:
            return int(float(leading_part))
        except ValueError:
            pass

    # --- Fallback: We could not interpret this value ---
    # Returning None lets the calling code identify and log problem rows.
    print(f"Warning: Could not convert value '{raw_value}' — returning None.")
    return None


# This function applies convert_likert_value() to an entire column in your DataFrame.
#
# Input:
#   dataframe      : your full pandas DataFrame
#   column_name    : the name of the column to clean, as a string
#   label_map      : the same label-to-integer dictionary passed to convert_likert_value()
#
# Returns:
#   A new pandas Series (a single column) with all values converted to integers or None.

In [72]:
def clean_all_likert_columns(dataframe, likert_columns, label_map):

    # We loop over each column name in the list.
    # On each iteration, "column_name" holds the current column's name as a string.
    for column_name in likert_columns:

        # Check that the column actually exists in the DataFrame before trying to clean it.
        # This prevents a confusing KeyError if there is a typo in your column list.
        if column_name not in dataframe.columns:
            print(f"Warning: Column '{column_name}' not found in DataFrame — skipping.")
            continue  # "continue" skips the rest of this iteration and moves to the next column

        # Apply the existing clean_likert_column() function to this column.
        # The result overwrites the original column in the DataFrame.
        dataframe[column_name] = clean_likert_column(dataframe, column_name, label_map)

        # Confirm each column as it is cleaned so you can spot problems early.
        print(f"Cleaned: '{column_name}'")

    return dataframe

In [79]:
uploader = widgets.FileUpload(accept=".csv", multiple=False)
display(uploader)

FileUpload(value=(), accept='.csv', description='Upload')

In [89]:
uploaded_file = uploader.value[0]
file_csv = io.BytesIO(uploaded_file["content"])

In [90]:
survey_table = pd.read_csv(file_csv)

In [91]:
display(survey_table.head(1))

,What are your general observations/impressions about the Batangas State University institutional visit?,What AI tools/technology or facilities from the Batangas State University stood out as relevant to your office's work or the Academy's offerings?,"What potential activities, practices, or approaches from the visit can be adopted or adapted for our programs and offerings?","What possible areas of collaboration with the Batangas State University do you see (e.g. training, research, joint projects, access to facilities)?",Activity Logistics.Food,Activity Logistics.Transportation,Activity Logistics.Pre-Training Arrangements/Coordination,"Rate your overall experience of the Institutional Visit. (1-Poor, 5-Excellent)",Any additional recommendations or ideas for the next steps?
0,My overall impression of the institutional vis...,The AI tools and facilities that stood out wer...,Their approach to showcasing applied research ...,Access to their specialized laboratories and f...,5,5,5,5,A recommended next step is to formalize the po...


In [92]:
likert_map = {
    "Very Dissatisfied": 1,
    "Dissatisfied":      2,
    "Neutral":           3,
    "Satisfied":         4,
    "Very Satisfied":    5
}
cleaned_df = clean_all_likert_columns(survey_table,survey_table.columns[9:23].to_list(),likert_map)

In [93]:
display(cleaned_df.head(10))

,What are your general observations/impressions about the Batangas State University institutional visit?,What AI tools/technology or facilities from the Batangas State University stood out as relevant to your office's work or the Academy's offerings?,"What potential activities, practices, or approaches from the visit can be adopted or adapted for our programs and offerings?","What possible areas of collaboration with the Batangas State University do you see (e.g. training, research, joint projects, access to facilities)?",Activity Logistics.Food,Activity Logistics.Transportation,Activity Logistics.Pre-Training Arrangements/Coordination,"Rate your overall experience of the Institutional Visit. (1-Poor, 5-Excellent)",Any additional recommendations or ideas for the next steps?
0,My overall impression of the institutional vis...,The AI tools and facilities that stood out wer...,Their approach to showcasing applied research ...,Access to their specialized laboratories and f...,5,5,5,5,A recommended next step is to formalize the po...
1,My overall impression of the Batangas State Un...,The AI tools and technology that stood out the...,"We can adopt some of their internal practices,...",Possible areas of collaboration include joint ...,4,4,4,4,Thank you for the invitation. I look forward t...
2,They possessed a clear and compelling vision f...,"The tools they use, along with the processes a...",Perhaps we can gather the geotags of our clien...,Possible areas of collaboration could include ...,4,5,4,4,"Perhaps for the next institutional visit, a pa..."
3,They have made significant investments in thei...,The Digital Transformation Center plays a pivo...,I am particularly interested in their microcre...,Training and access to facilities.,5,5,5,5,"Maybe, schedule a meeting dedicated to sharing..."
4,The BatStateU demonstrates strong leadership i...,N-CAIR and CAIST are most relevant to DAP for ...,The DAP can learn from BatStateU by setting up...,The DAP can work with BatStateU on training pr...,4,4,3,5,"Partnership with BatStateU, piloting joint AI ..."
5,BatStateU is widely recognized as a leading en...,BatStateU has been actively developing Artific...,Since the Philippines’ comprehensive AI legisl...,"If the DAP conducts training on AI, they can i...",4,5,3,4,NaN
6,It was informative.,IT Best Practices,Use of Open Source program for IT projects,ICT and AI,4,4,4,4,More institutional visits.
7,Leadership/management support played a vital r...,"Simulation System, SIGAW and LIKHA Fab Lab",Assistance/extension services to LGUs or commu...,technical training and access to facilities,4,4,3,4,Study visit may be extended


In [94]:
cleaned_df.to_csv("sample-survey-cleaned-2.csv")